# Definining Cohort

**The goal of this notebook is to identify patients with metastatic colorectal cancer who received first-line treatment with pembrolizumab or chemotherapy.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

In [3]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
df.head(3)

,PatientID,LineName,LineNumber,LineSetting,IsMaintenanceTherapy,EnhancedCohort,StartDate,EndDate
0,F80487DEC2205,FOLFOX,1,METASTATIC,False,MetastaticCRC,2024-12-10,2025-02-06
1,F80487DEC2205,"FOLFOX,Nivolumab",2,METASTATIC,False,MetastaticCRC,2025-02-07,2025-05-27
2,F80487DEC2205,Fluorouracil,2,METASTATIC,True,MetastaticCRC,2025-05-28,2025-09-15


In [5]:
df.EnhancedCohort.value_counts(dropna = False)

EnhancedCohort
MetastaticCRC    90183
Name: count, dtype: int64

In [6]:
df.IsMaintenanceTherapy.value_counts(dropna = False)

IsMaintenanceTherapy
False    80500
True      9683
Name: count, dtype: int64

In [7]:
df.LineSetting.value_counts(dropna = False)

LineSetting
METASTATIC    90183
Name: count, dtype: int64

In [8]:
df.query('LineNumber == 1').LineName.value_counts().head(30)

LineName
FOLFOX,Bevacizumab                          6599
FOLFOX                                      5778
Capecitabine                                4718
FOLFOX,Bevacizumab-Awwb                     2493
FOLFIRI,Bevacizumab                         2360
Fluorouracil,Leucovorin,Bevacizumab         1684
FOLFIRI                                     1270
Capecitabine,Bevacizumab                    1166
CAPEOX                                      1090
FOLFOX,Bevacizumab-Bvzr                     1089
Fluorouracil,Leucovorin                     1071
FOLFIRI,Bevacizumab-Awwb                     952
CAPEOX,Bevacizumab                           827
FOLFOX,Panitumumab                           774
Clinical Study Drug                          685
Fluorouracil,Leucovorin,Bevacizumab-Awwb     680
Fluorouracil                                 608
Pembrolizumab                                599
FOLFOXIRI                                    507
Capecitabine,Bevacizumab-Awwb                451
FOLFIRI,Cet

In [9]:
df.query('IsMaintenanceTherapy == True').LineName.value_counts().head(30)

LineName
Capecitabine                                1853
Fluorouracil,Leucovorin,Bevacizumab         1598
Capecitabine,Bevacizumab                     991
Fluorouracil,Leucovorin,Bevacizumab-Awwb     746
Fluorouracil,Leucovorin                      746
Capecitabine,Bevacizumab-Awwb                466
Fluorouracil,Leucovorin,Bevacizumab-Bvzr     323
Fluorouracil,Bevacizumab                     305
Fluorouracil,Leucovorin,Panitumumab          297
Bevacizumab                                  274
Cetuximab                                    231
Fluorouracil                                 205
Capecitabine,Bevacizumab-Bvzr                202
Fluorouracil,Leucovorin,Cetuximab            169
Panitumumab                                  166
Capecitabine,Panitumumab                     128
Capecitabine,Cetuximab                       104
Fluorouracil,Bevacizumab-Awwb                 93
Bevacizumab-Awwb                              75
Fluorouracil,Panitumumab                      61
Bevacizumab

## Pembrolizumab

In [10]:
pembro_df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Pembrolizumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro')
)

In [11]:
row_ID(pembro_df)

(594, 594)

## Chemotherapy

In [12]:
chemo_regimens = [
    # FOLFOX-based
    'FOLFOX', 
    'FOLFOX,Bevacizumab',
    'FOLFOX,Bevacizumab-Awwb',
    'FOLFOX,Bevacizumab-Bvzr',
    'FOLFOX,Cetuximab',

    # FOLFOXIRI-based
    'FOLFOXIRI', 
    'FOLFOXIRI,Bevacizumab',
    'FOLFOXIRI,Bevacizumab-Awwb',
    'FOLFOXIRI,Bevacizumab-Bvzr',
    'FOLFOXIRI,Cetuximab',
    
    # FOLFIRI-based
    'FOLFIRI',
    'FOLFIRI,Bevacizumab',
    'FOLFIRI,Bevacizumab-Awwb',
    'FOLFIRI,Bevacizumab-Bvzr',
    'FOLFIRI,Cetuximab', 
    
    # CAPEOX-based
    'CAPEOX',
    'CAPEOX,Bevacizumab',
    'CAPEOX,Bevacizumab-Awwb',
    'CAPEOX,Bevacizumab-Bvzr',
    'CAPEOX,Cetuximab',
    
    # Fluoropyrimidine-based
    'Fluorouracil,Leucovorin',
    'Fluorouracil,Leucovorin,Bevacizumab',
    'Fluorouracil,Leucovorin,Bevacizumab-Awwb',
    'Fluorouracil,Leucovorin,Bevacizumab-Bvzr',
    'Fluorouracil,Bevacizumab',
    'Fluorouracil,Leucovorin,Cetuximab',
]

chemo_df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == @chemo_regimens')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'chemo')
)

In [13]:
row_ID(chemo_df)

(26559, 25811)

## Remove duplicate IDs in chemotherapy dataframe

In [14]:
chemo_df['StartDate'] = pd.to_datetime(chemo_df['StartDate'])

In [15]:
chemo_df = (
    chemo_df
    .sort_values(['PatientID', 'StartDate'], ascending = True)
    .drop_duplicates(subset='PatientID', keep='first')
)

In [16]:
row_ID(chemo_df)

(25811, 25811)

## Merge and export to CSV

In [17]:
pembro_chemo_df = pd.concat([pembro_df, chemo_df], axis = 0)

In [18]:
row_ID(pembro_chemo_df)

(26405, 26405)

In [19]:
pembro_chemo_df.LineName.value_counts()

LineName
chemo     25811
pembro      594
Name: count, dtype: int64

In [20]:
pembro_chemo_df.to_csv('../outputs/pembro_chemo_index.csv', index = False)